# Wind Power Forecast Error Analysis (January 2024)

This notebook analyzes the accuracy of UK national wind power generation forecasts for January 2024 using data from Elexon BMRS.

## Objectives
1. Compute forecast error metrics (RMSE, Mean, Median, P99).
2. Analyze error distribution across different forecast horizons.
3. Investigate error patterns relative to the time of day.
4. Perform wind reliability analysis (P10, P50, P90).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from datetime import datetime, timedelta

plt.style.use('seaborn-v0_8')
BASE_URL = 'https://data.elexon.co.uk/bmrs/api/v1'

## 1. Data Ingestion
We fetch the actual generation and forecast data specifically for January 2024 as per instructions.

In [ ]:
def fetch_data():
    start_str = '2024-01-01'
    end_str = '2024-01-31'
    
    print(f"Fetching January 2024 data...")
    
    actual_url = f"{BASE_URL}/datasets/FUELHH/stream?startTimeFrom={start_str}&startTimeTo={end_str}"
    actuals = requests.get(actual_url).json()
    actuals_df = pd.DataFrame([d for d in actuals if d['fuelType'] == 'WIND'])
    
    forecast_url = f"{BASE_URL}/datasets/WINDFOR/stream?startTimeFrom={start_str}&startTimeTo={end_str}"
    forecasts = requests.get(forecast_url).json()
    forecasts_df = pd.DataFrame(forecasts)
    
    return actuals_df, forecasts_df

actuals_raw, forecasts_raw = fetch_data()

## 2. Forecast selection logic
We implement the horizon selection rule: `publishTime <= targetTime - horizon`.

In [ ]:
def process_data(actuals_df, forecasts_df, horizon_hours):
    f_df = forecasts_df.copy()
    a_df = actuals_df.copy()
    
    f_df['startTime'] = pd.to_datetime(f_df['startTime'])
    f_df['publishTime'] = pd.to_datetime(f_df['publishTime'])
    a_df['startTime'] = pd.to_datetime(a_df['startTime'])
    
    merged = []
    for _, actual in a_df.iterrows():
        target = actual['startTime']
        deadline = target - timedelta(hours=horizon_hours)
        
        valid = f_df[(f_df['startTime'] == target) & (f_df['publishTime'] <= deadline)]
        
        if not valid.empty:
            latest = valid.sort_values('publishTime', ascending=False).iloc[0]
            merged.append({
                'time': target,
                'actual': actual['generation'],
                'forecast': latest['generation'],
                'error': latest['generation'] - actual['generation']
            })
    return pd.DataFrame(merged)

df_4h = process_data(actuals_raw, forecasts_raw, 4)

## 3. Forecast Error Metrics
We calculate standard error metrics for the 4-hour horizon.

In [ ]:
def calculate_metrics(df):
    if df.empty: return pd.Series({'RMSE': 0, 'Mean Error': 0, 'Median Error': 0, 'P99 Error': 0})
    error = df['error']
    metrics = {
        'RMSE': np.sqrt(np.mean(error**2)),
        'Mean Error': np.mean(error),
        'Median Error': np.median(error),
        'P99 Error': np.percentile(np.abs(error), 99)
    }
    return pd.Series(metrics)

print("Metrics for 4-hour Horizon (January 2024):")
print(calculate_metrics(df_4h))

## 4. Error vs Forecast Horizon
We compare errors across 0, 4, 12, 24, and 48 hour horizons.

In [ ]:
horizons = [0, 4, 12, 24, 48]
horizon_results = []

for h in horizons:
    processed = process_data(actuals_raw, forecasts_raw, h)
    processed['horizon'] = h
    horizon_results.append(processed)

df_all_horizons = pd.concat(horizon_results)

if not df_all_horizons.empty:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='horizon', y='error', data=df_all_horizons)
    plt.title('Error Distribution vs. Forecast Horizon (January 2024)')
    plt.ylabel('Error (MW)')
    plt.xlabel('Horizon (Hours)')
    plt.show()
else:
    print("No data to plot for given horizons.")

## 5. Error vs Time of Day
Investigate if forecasts perform worse during specific periods (e.g., night vs day).

In [ ]:
if not df_4h.empty:
    df_4h['hour'] = df_4h['time'].dt.hour
    plt.figure(figsize=(12, 6))
    sns.lineplot(x='hour', y='error', data=df_4h, estimator='mean', errorbar='sd')
    plt.title('Mean Forecast Error by Hour of Day (4h Horizon, Jan 2024)')
    plt.ylabel('Mean Error (MW)')
    plt.grid(True)
    plt.show()

## 6. Wind Reliability Analysis
Analyze historical actual generation for January 2024 to determine reliable capacity.

In [ ]:
if not actuals_raw.empty:
    p10 = actuals_raw['generation'].quantile(0.1)
    p50 = actuals_raw['generation'].quantile(0.5)
    p90 = actuals_raw['generation'].quantile(0.9)

    print(f"P10 Generation: {p10:.2f} MW")
    print(f"P50 Generation: {p50:.2f} MW")
    print(f"P90 Generation: {p90:.2f} MW")

    print("\nRecommendation:")
    print(f"For January 2024, the reliable wind generation capacity should be based on the P10 value ({p10:.2f} MW),")
    print("as this level of generation is met or exceeded 90% of the time.")